<a href="https://colab.research.google.com/github/vansh-virmani/AIops-ca-2/blob/main/Bengaluru_House_price_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import re
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor


In [8]:


# Fixed: Enclosed the file path in quotes
df = pd.read_csv('/content/Bengaluru_House_Data.csv')

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (13320, 9)

Column names:
['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']

First 5 rows:


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00



Missing values:
area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [11]:



# Work on a copy
data = df.copy()

# Remove duplicate records
data = data.drop_duplicates()

# Convert area to a numeric value
def convert_sqft(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Handle ranges such as "2100 - 2850"
    match = re.match(
        r"^\s*([\d.]+)\s*-\s*([\d.]+)\s*$",
        value
    )

    if match:
        return (
            float(match.group(1)) +
            float(match.group(2))
        ) / 2

    # Handle ordinary numeric values
    try:
        return float(value)
    except ValueError:
        return np.nan

data["total_sqft"] = data["total_sqft"].apply(
    convert_sqft
)

# Convert "2 BHK", "4 Bedroom", etc. into numbers
data["bhk"] = pd.to_numeric(
    data["size"].astype(str).str.extract(
        r"(\d+(?:\.\d+)?)"
    )[0],
    errors="coerce"
)

# Convert bathroom count to numeric
data["bath"] = pd.to_numeric(
    data["bath"], errors="coerce"
)

# Convert target price to numeric
data["price"] = pd.to_numeric(
    data["price"], errors="coerce"
)

# Normalize location text
data["location"] = (
    data["location"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Remove records with missing essential fields
data = data.dropna(
    subset=["location", "total_sqft", "bhk", "price"]
)

# Remove invalid or nonpositive measurements
data = data[
    (data["total_sqft"] > 0) &
    (data["bhk"] > 0) &
    (data["price"] > 0)
].copy()

# Keep plausible property records for this assignment
data = data[
    data["total_sqft"] / data["bhk"] >= 200
].copy()

print("Shape after cleaning:", data.shape)
display(
    data[["location", "total_sqft", "bhk", "bath", "price"]]
    .head()
)
print("\nRemaining missing values:")
print(data[["location", "total_sqft", "bhk", "bath", "price"]].isnull().sum())


Shape after cleaning: (12389, 10)


,location,total_sqft,bhk,bath,price
0,Electronic City Phase II,1056.0,2.0,2.0,39.07
1,Chikka Tirupathi,2600.0,4.0,5.0,120.00
2,Uttarahalli,1440.0,3.0,2.0,62.00
3,Lingadheeranahalli,1521.0,3.0,3.0,95.00
4,Kothanur,1200.0,2.0,2.0,51.00



Remaining missing values:
location       0
total_sqft     0
bhk            0
bath          57
price          0
dtype: int64


In [12]:

feature_columns = [
    "location",
    "total_sqft",
    "bhk",
    "bath"
]

X = data[feature_columns].copy()
y = data["price"].copy()

print("Features:", X.columns.tolist())
print("Target:", y.name)
print("Number of samples:", len(X))


Features: ['location', 'total_sqft', 'bhk', 'bath']
Target: price
Number of samples: 12389


In [13]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 9911
Testing samples: 2478


In [15]:

numeric_features = [
    "total_sqft",
    "bhk",
    "bath"
]

categorical_features = ["location"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

print("Pipeline created successfully!")


Pipeline created successfully!


In [17]:
##Train the model

model.fit(X_train, y_train)

print("Model training completed!")


Model training completed!


In [18]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f} lakhs")
print(f"RMSE: {rmse:.2f} lakhs")
print(f"R²:   {r2:.4f}")


MAE:  34.06 lakhs
RMSE: 98.08 lakhs
R²:   0.5274


In [19]:

sample_house = pd.DataFrame([{
    "location": "Whitefield",
    "total_sqft": 1200,
    "bhk": 2,
    "bath": 2
}])

predicted_price = model.predict(sample_house)[0]

print(f"Predicted house price: ₹{predicted_price:.2f} lakhs")


Predicted house price: ₹48.28 lakhs


In [20]:

import joblib

joblib.dump(model, "model.joblib")

print("Saved model.joblib successfully!")


Saved model.joblib successfully!


In [21]:

from google.colab import files

files.download("model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
import sklearn
print(sklearn.__version__)

1.6.1
